# Semantic Search with Transformers

This notebook builds a semantic search engine over arXiv paper summaries.

Goal:
- Convert paper summaries into dense vector embeddings using a Transformer model.
- Index those vectors with FAISS for fast nearest-neighbor retrieval.
- Query the index with either an existing summary or a free-text prompt.

## Architecture Overview

Pipeline architecture:
1. Data Layer: Load and clean arXiv metadata and summaries.
2. Embedding Layer: Use SentenceTransformer (DistilBERT variant) to map each summary into a 768-dimensional vector.
3. Storage Layer: Persist vectors with pickle for reuse without recomputation.
4. Index Layer: Build a FAISS index over embeddings for efficient similarity search.
5. Retrieval Layer: Encode a query, search top-k nearest vectors, and map results back to paper titles/summaries.

Why this architecture works:
- Transformer embeddings capture semantic meaning beyond keyword overlap.
- FAISS enables fast vector search at scale compared to brute-force loops.
- Persisted embeddings separate expensive preprocessing from online querying.

## Architecture Diagram (PlantUML)

PlantUML source file used to generate the diagram:
- `architecture-diagram.puml`

Generated images:
- PNG version:

![Semantic Search Architecture PNG](architecture-diagram.png)

- JPG version:

![Semantic Search Architecture JPG](architecture-diagram.jpg)

## Official Resources

Model and embedding stack:
- Sentence-Transformers model card (`distilbert-base-nli-stsb-mean-tokens`): https://huggingface.co/sentence-transformers/distilbert-base-nli-stsb-mean-tokens
- SentenceTransformer API docs: https://www.sbert.net/docs/package_reference/SentenceTransformer.html
- sentence-transformers GitHub: https://github.com/UKPLab/sentence-transformers

Vector search:
- FAISS official docs: https://faiss.ai/
- FAISS GitHub: https://github.com/facebookresearch/faiss
- FAISS getting started wiki: https://github.com/facebookresearch/faiss/wiki/Getting-started

Core libraries used in notebook:
- Pandas docs (`read_json`): https://pandas.pydata.org/docs/reference/api/pandas.read_json.html
- scikit-learn `LabelEncoder`: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html
- Python `pickle` docs: https://docs.python.org/3/library/pickle.html
- PyTorch docs: https://pytorch.org/docs/stable/index.html

Diagram tooling:
- PlantUML: https://plantuml.com/
- Kroki (diagram rendering API): https://kroki.io/

In [1]:
# Run this cell beforehand so you do not see any warnings
import warnings
warnings.filterwarnings('ignore')

In [1]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn import preprocessing
import faiss
import numpy as np
import pickle

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load and Prepare Data

What happens in this step:
- Read the JSON dataset containing arXiv paper metadata.
- Remove columns not required for retrieval (`author`, `link`, `tag`).
- Keep core fields such as `id`, `title`, and `summary` for indexing and display.

Output:
- A cleaned DataFrame `df` used by all downstream steps.

In [4]:
data = pd.read_json('./arxivData.json')
df = data.drop(columns=["author", "link", "tag"])

print("No of papers" , df.id.unique().shape[0])

df.head()


No of papers 41000


,day,id,month,summary,title,year
0,1,1802.00209v1,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018
1,12,1603.03827v1,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016
2,2,1606.00776v2,6,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016
3,23,1705.08142v2,5,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017
4,7,1709.02349v2,9,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017


## Step 2: Load the Embedding Model

Model used:
- `distilbert-base-nli-stsb-mean-tokens` via `SentenceTransformer`.

Why this model:
- Produces sentence-level embeddings suitable for semantic similarity.
- Good trade-off between speed and quality for medium-size corpora.

Device handling:
- If CUDA is available, the model is moved to GPU for faster encoding.

In [5]:
model = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')

if torch.cuda.is_available():
    model = model.to(torch.device("cuda"))

print(model.device)

c:\Users\Admin\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--distilbert-base-nli-stsb-mean-tokens. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00,

cpu


## Step 3: Generate and Save Embeddings

What this step does:
- Extract all paper summaries from `df`.
- Use `model.encode(...)` to convert each summary into a dense vector.
- Save vectors to `embeddings.pkl` using `pickle.dump(...)`.
- Save corresponding paper IDs to `paper_ids.pkl` for mapping search results to metadata.

Why save to disk:
- Embedding generation is the most expensive stage.
- Persisting vectors avoids recomputation across sessions.

In [6]:
# Extract summaries from the dataframe
summaries = df['summary'].tolist()

print(f"Total number of summaries: {len(summaries)}")
print(f"\nFirst summary preview:\n{summaries[0][:200]}...")

# Encode all summaries to generate embeddings
print("\nGenerating embeddings using DistilBERT model...")
embeddings = model.encode(summaries, show_progress_bar=True, convert_to_numpy=True)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Each embedding dimension: {embeddings.shape[1]}")

# Save embeddings using pickle
embeddings_file = './embeddings.pkl'
with open(embeddings_file, 'wb') as f:
    pickle.dump(embeddings, f)

print(f"\nEmbeddings successfully saved to '{embeddings_file}'")

# Also save the paper IDs for reference (to map embeddings back to papers)
ids_file = './paper_ids.pkl'
paper_ids = df['id'].tolist()
with open(ids_file, 'wb') as f:
    pickle.dump(paper_ids, f)

print(f"Paper IDs successfully saved to '{ids_file}'")

Total number of summaries: 41000

First summary preview:
We propose an architecture for VQA which utilizes recurrent layers to
generate visual and textual attention. The memory characteristic of the
proposed recurrent attention units offers a rich joint emb...

Generating embeddings using DistilBERT model...


Batches: 100%|██████████| 1282/1282 [7:50:57<00:00, 22.04s/it]      



Embeddings shape: (41000, 768)
Each embedding dimension: 768

Embeddings successfully saved to './embeddings.pkl'
Paper IDs successfully saved to './paper_ids.pkl'


## Step 4: Load Persisted Embeddings

Purpose:
- Reload precomputed embeddings from disk so the search system can be rebuilt quickly.

Robustness detail:
- The loader checks for both `embeddings.pkl` and `embeddings.pickle` to avoid filename mismatch issues.

In [10]:
from pathlib import Path

candidate_files = [Path('embeddings.pkl'), Path('embeddings.pickle')]
emb_path = next((p for p in candidate_files if p.exists()), None)

if emb_path is None:
    raise FileNotFoundError(
        f"Could not find embeddings file. Tried: {[str(p) for p in candidate_files]}"
    )

with open(emb_path, 'rb') as pkl:
    embeddings = pickle.load(pkl)

print(f"Loaded embeddings from {emb_path}")
print(f"Embeddings shape: {embeddings.shape}")

Loaded embeddings from embeddings.pkl
Embeddings shape: (41000, 768)


In [12]:
length = len(embeddings)
print('Shape of the one embedding: ', embeddings[0].shape)

Shape of the one embedding:  (768,)


## Step 5: Data Preparation for Retrieval

Why label encoding is needed:
- FAISS `IndexIDMap` stores numeric IDs with vectors.
- Original paper IDs are strings (for example, `1802.00209v1`), so they are transformed to integer labels.

Helper method:
- `id2info(...)` converts retrieved integer IDs back to useful fields (`title`, `summary`) for readable output.

In [13]:
le = preprocessing.LabelEncoder()
df['id'] = le.fit_transform(df['id'])
df.head()


,day,id,month,summary,title,year
0,1,36693,2,We propose an architecture for VQA which utili...,Dual Recurrent Attention Units for Visual Ques...,2018
1,12,18198,3,Recent approaches based on artificial neural n...,Sequential Short-Text Classification with Recu...,2016
2,2,19318,6,We introduce the multiresolution recurrent neu...,Multiresolution Recurrent Neural Networks: An ...,2016
3,23,27779,5,Multi-task learning is motivated by the observ...,Learning what to share between loosely related...,2017
4,7,31468,9,We present MILABOT: a deep reinforcement learn...,A Deep Reinforcement Learning Chatbot,2017


In [14]:
def id2info(df, I, column):
    return [list(df[df.id == idx][column]) for idx in I]

## Step 6: Build the FAISS Index

FAISS details:
- FAISS (Facebook AI Similarity Search) is a library for efficient vector similarity search.
- It is optimized for high-dimensional dense vectors and large-scale nearest-neighbor retrieval.

Index used in this notebook:
- `faiss.IndexFlatL2(d)`: exact search with Euclidean (L2) distance in `d` dimensions.
- `faiss.IndexIDMap(...)`: wraps the base index so each vector is stored with an explicit integer ID.

Important implementation notes:
- Embeddings are converted to `float32` because FAISS expects this dtype.
- `index.add_with_ids(vectors, ids)` inserts vectors and preserves ID mapping.

Complexity intuition:
- `IndexFlatL2` performs exact comparisons against all vectors (high accuracy, slower at very large scale).
- For much larger datasets, approximate indexes (IVF, HNSW, PQ) can improve speed-memory trade-offs.

In [16]:
embeddings = np.array(embeddings).astype("float32")
index = faiss.IndexFlatL2(embeddings.shape[1])
index = faiss.IndexIDMap(index)
index.add_with_ids(embeddings, df['id'][:length])

print("Number of embeddings in Faiss Index", index.ntotal)

Number of embeddings in Faiss Index 41000


## Step 7: Similarity Search Using an Existing Summary

What happens:
- Pick one known summary embedding from the corpus.
- Query FAISS for top-k nearest vectors.
- Display distances, IDs, titles, and summaries of retrieved papers.

Interpretation:
- Lower L2 distance means higher similarity in embedding space.

In [17]:
df.iloc[1337, [3, 1]]

summary    In this paper we study the application of conv...
id                                                     12964
Name: 1337, dtype: object

In [18]:
D, I = index.search(np.array([embeddings[1337]]), k=10)
pd.DataFrame({'L2 distance': D.flatten().tolist(), 'ML paper IDs': I.flatten().tolist(), 'ML paper titles': id2info(df, I.flatten(), 'title'), 'Summaries': id2info(df, I.flatten(), 'summary')}).head(10)

,L2 distance,ML paper IDs,ML paper titles,Summaries
0,0.000000,12964,[Convolutional Neural Networks for joint objec...,[In this paper we study the application of con...
1,46.787518,23635,[Beyond Holistic Object Recognition: Enriching...,[Important high-level vision tasks such as hum...
2,55.466732,15599,[Maximum-Margin Structured Learning with Deep ...,[This paper focuses on structured-output learn...
3,56.263084,11454,[Articulated Pose Estimation by a Graphical Mo...,[We present a method for estimating articulate...
4,57.303955,36862,[Geometry-Contrastive Generative Adversarial N...,"[In this paper, we propose a geometry-contrast..."
5,58.229553,30524,[An Error Detection and Correction Framework f...,[We define and study error detection and corre...
6,58.414696,11720,[Object Segmentation in Images using EEG Signals],[This paper explores the potential of brain-co...
7,58.633751,31228,[Context Based Visual Content Verification],[In this paper the intermediary visual content...
8,59.194317,38637,[Learning Local Distortion Visibility From Ima...,[Accurate prediction of local distortion visib...
9,59.259262,17397,[Multi-Instance Visual-Semantic Embedding],[Visual-semantic embedding models have been re...


## Step 8: Similarity Search Using a Free-Text Prompt

Workflow:
- Define a natural-language user query.
- Encode the query with the same transformer model.
- Search top-k nearest papers in FAISS.
- Present matched titles and summaries as semantic recommendations.

Best-practice note:
- Encode the query as a single string input to preserve sentence meaning.
- Keep query embedding shape compatible with FAISS (`[1, d]`, `float32`).

In [19]:
user_query = "The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data"

In [20]:
embed = model.encode(list(user_query))
D, I = index.search(np.array([embed]).squeeze().astype("float32"), k=10)

results = {'L2 distances':D.flatten().tolist(), 'ML paper IDs':I.flatten().tolist(), "Titles": id2info(df, I.flatten(), 'title'), "Summaries": id2info(df, I.flatten(), 'summary')}

pd.DataFrame(results).head(10)

,L2 distances,ML paper IDs,Titles,Summaries
0,311.728455,19889,"[Neutrosophic Overset, Neutrosophic Underset, ...",[Neutrosophic Over-/Under-/Off-Set and -Logic ...
1,337.122925,832,[Using SLP Neural Network to Persian Handwritt...,[This paper has been withdrawn by the author a...
2,337.122925,863,[Recognition of Regular Shapes in Satelite Ima...,[This paper has been withdrawn by the author a...
3,344.939606,2195,[Convolutional Matching Pursuit and Dictionary...,[Matching pursuit and K-SVD is demonstrated in...
4,346.090179,30192,[Superposition de calques monochromes d'opacit...,[For a monochrome layer $x$ of opacity $0\le o...
5,346.649414,8365,[Segmentation et Interprétation de Nuages de P...,"[Dans cet article, nous pr\'esentons une m\'et..."
6,351.332275,487,[Utilisation des grammaires probabilistes dans...,[Nous pr\'esentons dans cette contribution une...
7,352.292786,6325,[On Information Regularization],[We formulate a principle for classification w...
8,353.913269,12981,[Inference for Sparse Conditional Precision Ma...,[Given $n$ i.i.d. observations of a random vec...
9,354.900024,6017,[The origin of Mayan languages from Formosan l...,[Basic body-part names (BBPNs) were defined as...


## Summary and Next Improvements

You have implemented an end-to-end semantic search system:
- Data ingestion and cleanup
- Embedding generation and persistence
- FAISS-based indexing
- Retrieval from summary example and free-text query

Potential improvements:
1. Use cosine similarity (normalize embeddings + inner product index) for better semantic ranking in many NLP tasks.
2. Persist the FAISS index itself to disk to skip rebuild time.
3. Add query preprocessing and shorter prompt variants for consistent retrieval quality.
4. Evaluate retrieval quality with manual checks or benchmark metrics (Precision@k, Recall@k).